In [5]:
!pip install torch torchvision streamlit pyngrok pillow numpy matplotlib

In [6]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# ======================
# VAE Model
# ======================
class VAE(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc1 = nn.Linear(784, 400)
        self.fc_mu = nn.Linear(400, 20)
        self.fc_logvar = nn.Linear(400, 20)
        self.fc2 = nn.Linear(20, 400)
        self.fc3 = nn.Linear(400, 784)

    def encode(self, x):
        h = torch.relu(self.fc1(x))
        return self.fc_mu(h), self.fc_logvar(h)

    def reparameterize(self, mu, logvar):
        std = torch.exp(0.5 * logvar)
        eps = torch.randn_like(std)
        return mu + eps * std

    def decode(self, z):
        h = torch.relu(self.fc2(z))
        return torch.sigmoid(self.fc3(h))

    def forward(self, x):
        mu, logvar = self.encode(x)
        z = self.reparameterize(mu, logvar)
        return self.decode(z), mu, logvar

# ======================
# Data
# ======================
transform = transforms.ToTensor()
dataset = datasets.MNIST(root="./data", train=True, download=True, transform=transform)
loader = DataLoader(dataset, batch_size=128, shuffle=True)

# ======================
# Train
# ======================
model = VAE().to(device)
optimizer = optim.Adam(model.parameters(), lr=1e-3)

def loss_fn(recon_x, x, mu, logvar):
    BCE = torch.nn.functional.binary_cross_entropy(recon_x, x.view(-1,784), reduction='sum')
    KLD = -0.5 * torch.sum(1 + logvar - mu.pow(2) - logvar.exp())
    return BCE + KLD

print("🚀 Training...")
for epoch in range(3):
    total_loss = 0
    for x,_ in loader:
        x = x.view(-1,784).to(device)

        recon, mu, logvar = model(x)
        loss = loss_fn(recon, x, mu, logvar)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    print(f"Epoch {epoch} Loss {total_loss:.2f}")

# حفظ الموديل
torch.save(model.state_dict(), "vae.pth")
print("✅ Model saved")

100%|██████████| 9.91M/9.91M [00:00<00:00, 18.4MB/s]
100%|██████████| 28.9k/28.9k [00:00<00:00, 502kB/s]
100%|██████████| 1.65M/1.65M [00:00<00:00, 4.65MB/s]
100%|██████████| 4.54k/4.54k [00:00<00:00, 8.96MB/s]


🚀 Training...
Epoch 0 Loss 9916901.64
Epoch 1 Loss 7310156.58
Epoch 2 Loss 6883633.71
✅ Model saved


In [7]:
%%writefile app.py
import streamlit as st
import torch
import torch.nn as nn
import numpy as np
from PIL import Image

# ======================
# VAE Model
# ======================
class VAE(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc2 = nn.Linear(20, 400)
        self.fc3 = nn.Linear(400, 784)

    def decode(self, z):
        h = torch.relu(self.fc2(z))
        return torch.sigmoid(self.fc3(h))

# تحميل الموديل
model = VAE()
model.load_state_dict(torch.load("vae.pth", map_location="cpu"), strict=False)
model.eval()

# ======================
# UI
# ======================
st.set_page_config(page_title="DigitGen", layout="centered")

st.title("🔥 DigitGen – VAE Generator")

st.write("تحكم في Latent Space وولّد أرقام")

# sliders
z = []
for i in range(20):
    val = st.slider(f"z[{i}]", -3.0, 3.0, 0.0)
    z.append(val)

# زر توليد
if st.button("Generate Digit"):
    z = torch.tensor(z).float().unsqueeze(0)

    with torch.no_grad():
        img = model.decode(z)

    img = img.numpy().reshape(28,28)
    img = (img * 255).astype(np.uint8)

    st.image(Image.fromarray(img), width=200)

Overwriting app.py


In [13]:
from pyngrok import ngrok

ngrok.set_auth_token("3BPVRViNhwWXwS5xUGa9qzCmyJr_3ZBUWGJmEE2cSv8nyyPBW")

In [ ]:
from pyngrok import ngrok

# فتح الرابط
public_url = ngrok.connect(8501)
print("🔥 افتح اللينك ده:", public_url)

# تشغيل التطبيق
!streamlit run app.py &

🔥 افتح اللينك ده: NgrokTunnel: "https://2148-34-16-192-159.ngrok-free.app" -> "http://localhost:8501"



  You can now view your Streamlit app in your browser.

  Local URL: http://localhost:8501
  Network URL: http://172.28.0.12:8501
  External URL: http://34.16.192.159:8501

